# Notebook 6: Real-Time Acoustic Analytics (Amplitude vs. Time & Spectrogram Engine)

This notebook demonstrates the high-precision **Acoustic Analytics & Diagnostic Engine** (`v1.5.0-rc1`), featuring:
- 📈 **Amplitude vs. Time:** Instantaneous physical Hilbert analytic envelopes ($A(t) = \sqrt{x^2 + \hat{x}^2}$) and Inter-aural Level Differences ($\Delta L(t)$ in dB).
- 📊 **Frequency vs. Time:** 2D Short-Time Fourier Transform (STFT) rolling waterfall spectrograms with Blackman-Harris windowing.
- 🎯 **Sub-Hertz Pitch Tracking:** Three-point parabolic fundamental frequency tracker ($f_0(t) \pm 0.2\,\text{Hz}$).
- ⏱ **Microsecond Phase Difference:** Instantaneous inter-channel phase monitoring ($\Delta \phi(t)$).

## 1. System Setup & Permissions
Ensure USB and bus access permissions are granted on the PYNQ-Z2 board.

In [ ]:
import numpy as np
from pynq_oscilloscope import check_usb_permissions, OscilloscopeOverlay, AcousticAnalytics

check_usb_permissions()

## 2. Load the Upgraded Hardware Overlay
Instantiate `OscilloscopeOverlay()`. It automatically loads the `v1.5.0-rc1` bitstream with simultaneous dual-ADC sampling and clean FFT demultiplexing.

In [ ]:
ol = OscilloscopeOverlay()
ol.set_profile("audio")
print("✅ Oscilloscope & Analytics Overlay loaded successfully!")
print(ol.get_profile_info())

## 3. Physical Verification: Zero-Skew Splitter Test
Verify true simultaneous sampling by feeding the same physical signal into both A0 and A1.

In [ ]:
# Capture synchronous stereo frame
v_a0, v_a1 = ol.capture_stereo()

# Calculate lag via cross-correlation
corr = np.correlate(v_a0 - np.mean(v_a0), v_a1 - np.mean(v_a1), mode='full')
lags = np.arange(-len(v_a0) + 1, len(v_a0))
delay_samples = lags[np.argmax(corr)]
delay_us = (delay_samples / ol.fs_per_ch) * 1e6

print(f"Measured Inter-Channel Delay: {delay_us:.3f} µs (Sample Lag: {delay_samples})")
if abs(delay_samples) == 0:
    print("🎯 PASSED: True Simultaneous Dual-ADC Sampling Confirmed (0.00 µs skew).")
else:
    print(f"ℹ️ Lag detected: {delay_samples} samples.")

## 4. Launch the Real-Time Diagnostic Dashboard
Launch the interactive Acoustic Analytics dashboard.

In [ ]:
# Render and launch real-time analytics
app = ol.analytic_dashboard()

## 5. Clean Shutdown
Halt live acquisition and release CMA memory.

In [ ]:
app.stop()
ol.close()
print("🔒 Analytics Dashboard stopped and memory released.")